# SEED-IV RD target-domain monitoring with DBCVT (diagnostic only)

This notebook trains the fold-specific RD representation with a structured EEG model:

`Dynamic Band Attention → Shared Channel Temporal Encoder → Channel-wise Voting → Channel Attention Fusion`.

The data pipeline and source-training normalization are unchanged. The model still receives the original flattened trial tensor from the existing CMRD dataloader, but internally reshapes it from `[B, T, 310]` to `[B, T, 62, 5]`.

It evaluates the held-out target subject every `TEST_EVERY` epochs and reports target accuracy as **mean ± population standard deviation across target subjects**.

> **Warning — target peeking:** changing hyperparameters after looking at these target curves leaks test-domain information. Use this notebook to diagnose model behavior, not to produce the final paper number. The official `scripts/train_rd.py` path remains source-validation-only.


In [1]:
from __future__ import annotations

import gc
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'cmrd').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook inside the CMRD repository')

ROOT = find_project_root(Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from cmrd.config import load_config
from cmrd.processed import load_split
from cmrd.training.engine import SequenceDataset, collate_sequences, evaluate, fit_normalizer
from cmrd.training.runtime import seed_everything

print('Project:', ROOT)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Project: C:\Users\Lin\Documents\Arbitruam\CMRD-Cute-Mew-Really-Delighting
PyTorch: 2.13.0.dev20260422+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 5080 Laptop GPU


## Parameters
Edit this cell between diagnostic runs. Change `RUN_TAG` whenever parameters change so results are not mixed. Start with a few subjects such as `[1, 2, 3]`; use all 15 only after the code and settings look right.

In [2]:
CONFIG_PATH = ROOT / 'configs' / 'seediv' / 'rd.yaml'
TARGET_SUBJECTS = list(range(1, 16))  # e.g. [1, 2, 3] for a quick check

# Structured model: Dynamic Band-Channel Voting Transformer (DBCVT)
MODEL_NAME = 'dbcv_transformer_v1'
CHANNELS = 62
BANDS = 5

# Temporal encoder inside each channel. Start from 2-3 layers; deeper settings should be swept.
D_MODEL = 192
NHEAD = 6
LAYERS = 3
FEEDFORWARD = 768
DROPOUT = 0.2

# Band attention and channel aggregation
BAND_ATTENTION_HIDDEN = 32
CHANNEL_ATTENTION_HIDDEN = 128
VOTE_LOSS_WEIGHT = 0.05  # set 0.0 to disable channel-wise auxiliary voting loss

# Optimization
EPOCHS = 100
TEST_EVERY = 10
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
MINIMUM_LEARNING_RATE = 1e-6
WEIGHT_DECAY = 1e-3
LABEL_SMOOTHING = 0.05
GRADIENT_CLIP_NORM = 1.0
SEED = 42
DETERMINISTIC = True
DEVICE = 'cuda'  # explicit for the RTX 5080 Laptop
NUM_WORKERS = 0  # safest setting for Windows + Jupyter

# Output / resume
RUN_TAG = 'dbcv_v1_d192_l3_drop0.3_wd1e-3_seed42'
RESUME = True
SAVE_CHECKPOINTS = True

assert EPOCHS > 0 and TEST_EVERY > 0
assert D_MODEL % NHEAD == 0
assert len(set(TARGET_SUBJECTS)) == len(TARGET_SUBJECTS)
assert all(1 <= subject <= 15 for subject in TARGET_SUBJECTS)
assert CHANNELS == int(load_config(CONFIG_PATH, expected_feature='rd').raw['dataset']['channels'])
assert BANDS == 5, 'This notebook assumes delta/theta/alpha/beta/gamma RD features.'
if DEVICE.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError('CUDA requested but unavailable')

RUN_DIR = ROOT / 'runs' / 'diagnostics' / 'seediv_rd_target_monitor' / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR


WindowsPath('C:/Users/Lin/Documents/Arbitruam/CMRD-Cute-Mew-Really-Delighting/runs/diagnostics/seediv_rd_target_monitor/dbcv_v1_d192_l3_drop0.3_wd1e-3_seed42')

In [3]:
CONFIG = load_config(CONFIG_PATH, expected_feature='rd')
DEVICE_OBJ = torch.device(DEVICE)

SETTINGS = {
    'config': str(CONFIG_PATH),
    'config_hash': CONFIG.hash(),
    'preprocessing_signature': CONFIG.preprocessing_signature(),
    'target_subjects': TARGET_SUBJECTS,
    'model': {
        'name': MODEL_NAME,
        'channels': CHANNELS,
        'bands': BANDS,
        'd_model': D_MODEL,
        'nhead': NHEAD,
        'layers': LAYERS,
        'feedforward': FEEDFORWARD,
        'dropout': DROPOUT,
        'band_attention_hidden': BAND_ATTENTION_HIDDEN,
        'channel_attention_hidden': CHANNEL_ATTENTION_HIDDEN,
        'vote_loss_weight': VOTE_LOSS_WEIGHT,
    },
    'training': {
        'epochs': EPOCHS, 'test_every': TEST_EVERY, 'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE, 'minimum_learning_rate': MINIMUM_LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY, 'label_smoothing': LABEL_SMOOTHING,
        'gradient_clip_norm': GRADIENT_CLIP_NORM, 'seed': SEED,
        'deterministic': DETERMINISTIC, 'device': DEVICE,
    },
    'diagnostic_target_peeking': True,
}
settings_path = RUN_DIR / 'settings.json'
if settings_path.exists():
    previous = json.loads(settings_path.read_text(encoding='utf-8'))
    if previous != SETTINGS:
        raise RuntimeError(
            f'RUN_TAG={RUN_TAG!r} already exists with different settings. '
            'Change RUN_TAG or delete the old diagnostic directory.'
        )
settings_path.write_text(json.dumps(SETTINGS, indent=2, ensure_ascii=False), encoding='utf-8')
SETTINGS


{'config': 'C:\\Users\\Lin\\Documents\\Arbitruam\\CMRD-Cute-Mew-Really-Delighting\\configs\\seediv\\rd.yaml',
 'config_hash': '8ca09d529479',
 'preprocessing_signature': 'a19fe089cb9d9481',
 'target_subjects': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
 'model': {'name': 'dbcv_transformer_v1',
  'channels': 62,
  'bands': 5,
  'd_model': 192,
  'nhead': 6,
  'layers': 3,
  'feedforward': 768,
  'dropout': 0.2,
  'band_attention_hidden': 32,
  'channel_attention_hidden': 128,
  'vote_loss_weight': 0.05},
 'training': {'epochs': 100,
  'test_every': 10,
  'batch_size': 4,
  'learning_rate': 0.0001,
  'minimum_learning_rate': 1e-06,
  'weight_decay': 0.001,
  'label_smoothing': 0.05,
  'gradient_clip_norm': 1.0,
  'seed': 42,
  'deterministic': True,
  'device': 'cuda'},
 'diagnostic_target_peeking': True}

## Training helpers
The source-training normalization and source-validation split are unchanged. The only deliberate protocol violation is evaluating the held-out target subject every `TEST_EVERY` epochs.

In [4]:
class SinusoidalPosition(nn.Module):
    def __init__(self, d_model: int, max_length: int) -> None:
        super().__init__()
        positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
        divisor = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10_000.0) / d_model))
        encoding = torch.zeros(max_length, d_model)
        encoding[:, 0::2] = torch.sin(positions * divisor)
        encoding[:, 1::2] = torch.cos(positions * divisor[: encoding[:, 1::2].shape[1]])
        self.register_buffer('encoding', encoding.unsqueeze(0), persistent=False)

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        return value + self.encoding[:, : value.shape[1]]


class DynamicBandChannelVotingTransformer(nn.Module):
    """
    DBCVT-v1.

    Input interface remains compatible with the existing CMRD loaders:
        data: [batch, time, channels * bands]
        mask: [batch, time]

    Internally:
        [B,T,C*F] -> [B,T,C,F]
        dynamic band attention -> [B,T,C,d_model]
        shared temporal Transformer per channel -> [B,C,d_model]
        channel-wise logits + channel attention fusion -> [B,num_classes]
    """

    def __init__(
        self,
        input_dim: int,
        classes: int,
        max_length: int,
        channels: int,
        bands: int,
        d_model: int,
        nhead: int,
        layers: int,
        feedforward: int,
        dropout: float,
        band_attention_hidden: int,
        channel_attention_hidden: int,
    ) -> None:
        super().__init__()
        if input_dim != channels * bands:
            raise ValueError(f'Expected input_dim={channels * bands} from channels*bands, got {input_dim}')
        if d_model % nhead != 0:
            raise ValueError(f'd_model={d_model} must be divisible by nhead={nhead}')

        self.input_dim = int(input_dim)
        self.classes = int(classes)
        self.channels = int(channels)
        self.bands = int(bands)
        self.d_model = int(d_model)

        # Dynamic frequency-band attention for each [sample,time,channel].
        self.band_gate = nn.Sequential(
            nn.LayerNorm(bands),
            nn.Linear(bands, band_attention_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(band_attention_hidden, bands),
        )
        self.band_embedding = nn.Parameter(torch.empty(bands, d_model))
        nn.init.xavier_uniform_(self.band_embedding)

        self.band_norm = nn.LayerNorm(d_model)
        self.channel_embedding = nn.Parameter(torch.zeros(channels, d_model))
        nn.init.normal_(self.channel_embedding, mean=0.0, std=0.02)
        self.input_dropout = nn.Dropout(dropout)

        # One shared temporal encoder is applied to every channel independently.
        self.position = SinusoidalPosition(d_model, max_length)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.temporal_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=layers,
            enable_nested_tensor=False,
        )

        # Each channel casts one vote.
        self.channel_vote_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, classes),
        )

        # A global module decides how much to trust each channel.
        self.channel_attention = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, channel_attention_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(channel_attention_hidden, 1),
        )

        # Representation-level fusion branch; it stabilizes pure logit voting.
        self.global_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, classes),
        )

    def forward(self, data: torch.Tensor, mask: torch.Tensor, return_aux: bool = False):
        if data.ndim != 3 or mask.shape != data.shape[:2]:
            raise ValueError(f'Expected data [B,T,F] and mask [B,T], got {data.shape}, {mask.shape}')
        if data.shape[-1] != self.input_dim:
            raise ValueError(f'Expected flattened feature dim {self.input_dim}, got {data.shape[-1]}')

        batch, time_steps, _ = data.shape
        x = data.view(batch, time_steps, self.channels, self.bands)  # [B,T,C,F]

        # Band attention is dynamic over sample, time, and channel.
        band_scores = self.band_gate(x)                       # [B,T,C,F]
        band_weights = torch.softmax(band_scores, dim=-1)     # [B,T,C,F]
        weighted_bands = x * band_weights

        # Convert 5 scalar band values into a d_model channel-token representation.
        channel_tokens = torch.einsum('btcf,fd->btcd', weighted_bands, self.band_embedding)
        channel_tokens = self.band_norm(channel_tokens)
        channel_tokens = channel_tokens + self.channel_embedding.view(1, 1, self.channels, self.d_model)
        channel_tokens = self.input_dropout(channel_tokens)

        # Encode each channel's temporal sequence with a shared Transformer.
        # [B,T,C,D] -> [B*C,T,D]
        per_channel = channel_tokens.permute(0, 2, 1, 3).reshape(batch * self.channels, time_steps, self.d_model)
        per_channel_mask = mask.unsqueeze(1).expand(batch, self.channels, time_steps).reshape(batch * self.channels, time_steps)

        encoded = self.position(per_channel)
        encoded = self.temporal_encoder(encoded, src_key_padding_mask=~per_channel_mask)

        weights_t = per_channel_mask.unsqueeze(-1).to(encoded.dtype)
        channel_repr = (encoded * weights_t).sum(dim=1) / weights_t.sum(dim=1).clamp_min(1.0)
        channel_repr = channel_repr.view(batch, self.channels, self.d_model)  # [B,C,D]

        channel_logits = self.channel_vote_head(channel_repr)                 # [B,C,K]
        channel_scores = self.channel_attention(channel_repr).squeeze(-1)     # [B,C]
        channel_weights = torch.softmax(channel_scores, dim=1)                # [B,C]

        vote_logits = (channel_logits * channel_weights.unsqueeze(-1)).sum(dim=1)
        global_repr = (channel_repr * channel_weights.unsqueeze(-1)).sum(dim=1)
        global_logits = self.global_head(global_repr)

        logits = vote_logits + global_logits

        if return_aux:
            return {
                'logits': logits,
                'channel_logits': channel_logits,
                'channel_weights': channel_weights,
                'band_weights': band_weights,
                'vote_logits': vote_logits,
                'global_logits': global_logits,
            }
        return logits


In [5]:
def make_loader(samples, mean, std, shuffle: bool, seed: int) -> DataLoader:
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        SequenceDataset(samples, mean, std),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE_OBJ.type == 'cuda',
        collate_fn=collate_sequences,
        generator=generator,
    )

def train_one_fold(target_subject: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    fold_epoch_path = RUN_DIR / f'fold-{target_subject:02d}_epochs.csv'
    fold_target_path = RUN_DIR / f'fold-{target_subject:02d}_target_curve.csv'
    if RESUME and fold_epoch_path.is_file() and fold_target_path.is_file():
        old_epochs = pd.read_csv(fold_epoch_path)
        old_target = pd.read_csv(fold_target_path)
        if len(old_epochs) == EPOCHS and int(old_epochs['epoch'].max()) == EPOCHS:
            print(f'fold={target_subject:02d} reused from disk')
            return old_epochs, old_target

    seed_everything(SEED, DETERMINISTIC)
    # Candidate settings are already fixed before target samples are loaded.
    train_samples, validation_samples, target_samples, split = load_split(CONFIG, target_subject, include_test=True)
    mean, std = fit_normalizer(train_samples)
    max_length = max(sample.x.shape[0] for sample in train_samples + validation_samples + target_samples)
    input_dim = train_samples[0].x.shape[1]

    train_loader = make_loader(train_samples, mean, std, True, SEED)
    validation_loader = make_loader(validation_samples, mean, std, False, SEED)
    target_loader = make_loader(target_samples, mean, std, False, SEED)
    if input_dim != CHANNELS * BANDS:
        raise ValueError(f'Expected RD feature_dim={CHANNELS * BANDS}, got {input_dim}. '
                         'Check CHANNELS/BANDS or the preprocessed cache.')

    model = DynamicBandChannelVotingTransformer(
        input_dim=input_dim,
        classes=int(CONFIG.raw['dataset']['classes']),
        max_length=max_length,
        channels=CHANNELS,
        bands=BANDS,
        d_model=D_MODEL,
        nhead=NHEAD,
        layers=LAYERS,
        feedforward=FEEDFORWARD,
        dropout=DROPOUT,
        band_attention_hidden=BAND_ATTENTION_HIDDEN,
        channel_attention_hidden=CHANNEL_ATTENTION_HIDDEN,
    ).to(DEVICE_OBJ)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MINIMUM_LEARNING_RATE)

    epoch_rows = []
    target_rows = []
    best_source_score = (-float('inf'), -float('inf'))
    best_source_state = None
    best_source_epoch = 0
    started = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        main_loss_sum = 0.0
        vote_loss_sum = 0.0
        seen = 0
        for data, mask, labels in train_loader:
            data = data.to(DEVICE_OBJ, non_blocking=True)
            mask = mask.to(DEVICE_OBJ, non_blocking=True)
            labels = labels.to(DEVICE_OBJ, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            output = model(data, mask, return_aux=True)
            main_loss = criterion(output['logits'], labels)
            vote_loss = torch.zeros((), device=DEVICE_OBJ)
            if VOTE_LOSS_WEIGHT > 0:
                labels_per_channel = labels.unsqueeze(1).expand(-1, CHANNELS).reshape(-1)
                vote_loss = criterion(output['channel_logits'].reshape(-1, int(CONFIG.raw['dataset']['classes'])), labels_per_channel)
            loss = main_loss + float(VOTE_LOSS_WEIGHT) * vote_loss
            loss.backward()
            if GRADIENT_CLIP_NORM > 0:
                nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
            optimizer.step()
            loss_sum += float(loss.item()) * labels.shape[0]
            main_loss_sum += float(main_loss.item()) * labels.shape[0]
            vote_loss_sum += float(vote_loss.item()) * labels.shape[0]
            seen += labels.shape[0]
        scheduler.step()

        source_metrics = evaluate(model, validation_loader, DEVICE_OBJ, int(CONFIG.raw['dataset']['classes']))
        source_score = (float(source_metrics['macro_f1']), float(source_metrics['accuracy']))
        if source_score > best_source_score:
            best_source_score = source_score
            best_source_epoch = epoch
            best_source_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

        row = {
            'target_subject': target_subject, 'epoch': epoch,
            'train_loss': loss_sum / max(seen, 1),
            'train_main_loss': main_loss_sum / max(seen, 1),
            'train_vote_loss': vote_loss_sum / max(seen, 1),
            'learning_rate': optimizer.param_groups[0]['lr'],
            'source_validation_accuracy': source_metrics['accuracy'],
            'source_validation_macro_f1': source_metrics['macro_f1'],
            'elapsed_seconds': time.perf_counter() - started,
        }
        epoch_rows.append(row)

        if epoch % TEST_EVERY == 0 or epoch == EPOCHS:
            target_metrics = evaluate(model, target_loader, DEVICE_OBJ, int(CONFIG.raw['dataset']['classes']))
            target_row = {
                **row,
                'target_accuracy': target_metrics['accuracy'],
                'target_balanced_accuracy': target_metrics['balanced_accuracy'],
                'target_macro_f1': target_metrics['macro_f1'],
                'target_confusion_matrix': json.dumps(target_metrics['confusion_matrix']),
            }
            target_rows.append(target_row)
            print(
                f'fold={target_subject:02d} epoch={epoch:03d} '
                f'loss={row["train_loss"]:.4f} source_val_f1={source_metrics["macro_f1"]:.4f} '
                f'target_acc={target_metrics["accuracy"]:.4f} target_f1={target_metrics["macro_f1"]:.4f}'
            )

    epoch_frame = pd.DataFrame(epoch_rows)
    target_frame = pd.DataFrame(target_rows)
    epoch_frame.to_csv(fold_epoch_path, index=False)
    target_frame.to_csv(fold_target_path, index=False)

    if SAVE_CHECKPOINTS and best_source_state is not None:
        torch.save(
            {
                'model_state_dict': best_source_state,
                'normalization_mean': mean, 'normalization_std': std,
                'best_source_validation_epoch': best_source_epoch,
                'best_source_validation_score': best_source_score,
                'target_subject': target_subject, 'split': split.as_dict(), 'settings': SETTINGS,
                'warning': 'Target was monitored during training; do not report as paper-final.',
            },
            RUN_DIR / f'fold-{target_subject:02d}_best-source-validation.pt',
        )

    del model, optimizer, scheduler, train_loader, validation_loader, target_loader
    del train_samples, validation_samples, target_samples
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return epoch_frame, target_frame


## Run selected LOSO folds
This trains one independent model per target subject. After each fold, the running table shows mean ± std across the target subjects completed so far. The final table is meaningful only when `n_subjects == 15`.

In [6]:
all_epoch_frames = []
all_target_frames = []

for target_subject in TARGET_SUBJECTS:
    epoch_frame, target_frame = train_one_fold(target_subject)
    all_epoch_frames.append(epoch_frame)
    all_target_frames.append(target_frame)
    running = pd.concat(all_target_frames, ignore_index=True)
    running_summary = (
        running.groupby('epoch', as_index=False)
        .agg(
            target_acc_mean=('target_accuracy', 'mean'),
            target_acc_std=('target_accuracy', lambda values: values.std(ddof=0)),
            target_macro_f1_mean=('target_macro_f1', 'mean'),
            target_macro_f1_std=('target_macro_f1', lambda values: values.std(ddof=0)),
            n_subjects=('target_subject', 'nunique'),
        )
    )
    latest = running_summary.iloc[-1]
    print(
        f'RUNNING epoch={int(latest.epoch):03d}: target ACC '
        f'{latest.target_acc_mean:.4f} ± {latest.target_acc_std:.4f} '
        f'across {int(latest.n_subjects)} subject(s)'
    )


fold=01 epoch=010 loss=1.1414 source_val_f1=0.4960 target_acc=0.5278 target_f1=0.4949
fold=01 epoch=020 loss=0.8774 source_val_f1=0.6177 target_acc=0.6528 target_f1=0.6496
fold=01 epoch=030 loss=0.7274 source_val_f1=0.6533 target_acc=0.6806 target_f1=0.6796
fold=01 epoch=040 loss=0.6182 source_val_f1=0.6928 target_acc=0.6667 target_f1=0.6600
fold=01 epoch=050 loss=0.5065 source_val_f1=0.6544 target_acc=0.6528 target_f1=0.6464
fold=01 epoch=060 loss=0.4585 source_val_f1=0.6593 target_acc=0.6250 target_f1=0.6229
fold=01 epoch=070 loss=0.3872 source_val_f1=0.6957 target_acc=0.6944 target_f1=0.6905
fold=01 epoch=080 loss=0.3583 source_val_f1=0.6882 target_acc=0.7361 target_f1=0.7340
fold=01 epoch=090 loss=0.3437 source_val_f1=0.7264 target_acc=0.7222 target_f1=0.7173
fold=01 epoch=100 loss=0.3453 source_val_f1=0.7201 target_acc=0.7222 target_f1=0.7173
RUNNING epoch=100: target ACC 0.7222 ± 0.0000 across 1 subject(s)
fold=02 epoch=010 loss=1.1449 source_val_f1=0.4949 target_acc=0.5139 targe

KeyboardInterrupt: 

## Aggregate every-10-epoch target curve

In [ ]:
epoch_results = pd.concat(all_epoch_frames, ignore_index=True)
target_results = pd.concat(all_target_frames, ignore_index=True)
target_summary = (
    target_results.groupby('epoch', as_index=False)
    .agg(
        target_acc_mean=('target_accuracy', 'mean'),
        target_acc_std=('target_accuracy', lambda values: values.std(ddof=0)),
        target_acc_min=('target_accuracy', 'min'),
        target_acc_max=('target_accuracy', 'max'),
        target_macro_f1_mean=('target_macro_f1', 'mean'),
        target_macro_f1_std=('target_macro_f1', lambda values: values.std(ddof=0)),
        source_validation_acc_mean=('source_validation_accuracy', 'mean'),
        source_validation_macro_f1_mean=('source_validation_macro_f1', 'mean'),
        n_subjects=('target_subject', 'nunique'),
    )
)

epoch_results.to_csv(RUN_DIR / 'all_fold_epochs.csv', index=False)
target_results.to_csv(RUN_DIR / 'all_fold_target_curve.csv', index=False)
target_summary.to_csv(RUN_DIR / 'target_mean_std_by_epoch.csv', index=False)

display(target_summary.style.format({
    'target_acc_mean': '{:.2%}', 'target_acc_std': '{:.2%}',
    'target_acc_min': '{:.2%}', 'target_acc_max': '{:.2%}',
    'target_macro_f1_mean': '{:.2%}', 'target_macro_f1_std': '{:.2%}',
    'source_validation_acc_mean': '{:.2%}', 'source_validation_macro_f1_mean': '{:.2%}',
}))

best_target_row = target_summary.loc[target_summary['target_acc_mean'].idxmax()]
print(
    f'Exploratory best target checkpoint: epoch={int(best_target_row.epoch)}, '
    f'ACC={best_target_row.target_acc_mean:.2%} ± {best_target_row.target_acc_std:.2%}; '
    'this epoch is target-selected and must not be reported as a clean test result.'
)


In [ ]:
try:
    import matplotlib.pyplot as plt

    figure, axis = plt.subplots(figsize=(9, 5))
    axis.plot(target_summary['epoch'], target_summary['target_acc_mean'], marker='o', label='Target ACC mean')
    axis.fill_between(
        target_summary['epoch'],
        target_summary['target_acc_mean'] - target_summary['target_acc_std'],
        target_summary['target_acc_mean'] + target_summary['target_acc_std'],
        alpha=0.2, label='± 1 subject std',
    )
    axis.plot(target_summary['epoch'], target_summary['source_validation_acc_mean'], linestyle='--', label='Source validation ACC mean')
    axis.set(xlabel='Epoch', ylabel='Accuracy', title='SEED-IV RD diagnostic target monitoring')
    axis.grid(alpha=0.3)
    axis.legend()
    figure.tight_layout()
    figure.savefig(RUN_DIR / 'target_accuracy_curve.png', dpi=160)
    plt.show()
except ModuleNotFoundError:
    print('matplotlib is not installed; CSV summaries were still saved.')


## How to interpret this DBCVT diagnostic run

If this model improves `source_val_f1` over the plain Transformer, the structured `[B,T,62,5]` inductive bias is useful.

If train loss still collapses to nearly zero while `source_val_f1` remains flat, the next likely issues are:

1. RD alone may be weaker than DE or DE+RD because it keeps relative spectral deviation but not absolute DE energy.
2. The model may still overfit source subjects; try stronger dropout, label smoothing, and smaller `D_MODEL`.
3. Check per-class confusion matrices. A 4-class SEED-IV fold can look acceptable in accuracy while one emotion class collapses.
4. Do not select hyperparameters from target curves for final reporting. This notebook deliberately monitors target only for diagnosis.

Suggested sweep after the first run:

- `LAYERS = 2 / 3 / 4`
- `D_MODEL = 128 / 192 / 256`
- `VOTE_LOSS_WEIGHT = 0.0 / 0.05 / 0.1`
- `DROPOUT = 0.3 / 0.4`
